# Streaming and Structured Output with Ollama

This notebook extends the basic local client with two integration patterns:

1. stream generated text so an interface can react before the full response is complete; and
2. request JSON that can be parsed and validated before another component uses it.

Both patterns improve application behavior, but neither guarantees factual or safe model output.

## Learning Goals

- read Ollama's newline-delimited streaming events;
- measure client-observed time to first content and total response time;
- request output using a JSON Schema;
- distinguish parsing, schema validation, and semantic correctness; and
- handle unavailable services and malformed responses explicitly.

In [ ]:
import json
import time
from collections.abc import Iterator
from typing import Any

import requests
from jsonschema import ValidationError, validate

## 1. Configure the Local API

The health check keeps the notebook runnable when Ollama is not installed. Live cells are skipped, while parsing and validation examples still execute.

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434"
GENERATE_URL = f"{OLLAMA_BASE_URL}/api/generate"
TAGS_URL = f"{OLLAMA_BASE_URL}/api/tags"
MODEL_NAME = "qwen3.5:2b"
REQUEST_TIMEOUT_SECONDS = 120
MAX_OUTPUT_TOKENS = 128

session = requests.Session()


def installed_model_names() -> list[str]:
    """Return installed Ollama model tags, or an empty list if unavailable."""
    try:
        response = session.get(TAGS_URL, timeout=3)
        response.raise_for_status()
    except requests.RequestException:
        return []
    return [
        item.get("name", "")
        for item in response.json().get("models", [])
        if item.get("name")
    ]


installed_models = installed_model_names()
model_available = any(name == MODEL_NAME for name in installed_models)
print(f"Installed models: {installed_models or 'Ollama unavailable or no models'}")
if not model_available:
    print(f"Live examples require: ollama pull {MODEL_NAME}")

## 2. Understand the Streaming Contract

With `stream: true`, Ollama sends newline-delimited JSON events. Content arrives in the `response` field of successive events. The final event has `done: true` and includes usage metrics.

![Ollama API usage fields included in the final event](assets/ollama-api-metrics.png)

*The final event carries the token counts and durations needed for evaluation. Source: [Ollama API usage documentation](https://docs.ollama.com/api/usage).*

A robust client must:

1. keep the HTTP connection open;
2. parse each non-empty line independently;
3. record the first event containing visible content;
4. concatenate content in arrival order; and
5. retain the final metrics event.

See the [official streaming documentation](https://docs.ollama.com/api/streaming) for the endpoint behavior.

In [ ]:
def parse_json_lines(lines: Iterator[bytes]) -> Iterator[dict[str, Any]]:
    """Parse non-empty newline-delimited JSON events."""
    for line in lines:
        if not line:
            continue
        event = json.loads(line)
        if not isinstance(event, dict):
            raise TypeError("Expected each streaming event to be a JSON object")
        yield event


fixed_lines = iter(
    [
        b'{"response":"Hello", "done":false}',
        b'{"response":" world", "done":false}',
        b'{"response":"", "done":true, "eval_count":2}',
    ]
)
fixed_events = list(parse_json_lines(fixed_lines))
assert "".join(event.get("response", "") for event in fixed_events) == "Hello world"
fixed_events

## 3. Measure Time to First Content

Time to first content is measured by the client clock. It starts immediately before the request and stops when the first non-empty generated fragment arrives. Total client time stops after the final event.

This is different from Ollama's internal `total_duration`: network and client processing can make the observed wall time slightly different, even on localhost. The example disables optional reasoning so the first measured content is the user-facing answer rather than a separate thinking stream.

In [ ]:
def stream_generate(
    prompt: str,
    *,
    model: str = MODEL_NAME,
    temperature: float = 0.2,
) -> dict[str, Any]:
    """Stream one response and return text, timing, and final API metrics."""
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": True,
        "think": False,
        "options": {
            "temperature": temperature,
            "num_predict": MAX_OUTPUT_TOKENS,
        },
    }

    started = time.perf_counter()
    first_content_seconds: float | None = None
    fragments: list[str] = []
    final_event: dict[str, Any] = {}

    with session.post(
        GENERATE_URL,
        json=payload,
        stream=True,
        timeout=REQUEST_TIMEOUT_SECONDS,
    ) as response:
        response.raise_for_status()
        for event in parse_json_lines(response.iter_lines()):
            fragment = str(event.get("response", ""))
            if fragment and first_content_seconds is None:
                first_content_seconds = time.perf_counter() - started
            fragments.append(fragment)
            if event.get("done") is True:
                final_event = event

    total_wall_seconds = time.perf_counter() - started
    return {
        "text": "".join(fragments),
        "first_content_seconds": first_content_seconds,
        "total_wall_seconds": total_wall_seconds,
        "eval_count": int(final_event.get("eval_count", 0)),
        "eval_duration_ns": int(final_event.get("eval_duration", 0)),
        "done_reason": final_event.get("done_reason"),
    }

In [ ]:
stream_result: dict[str, Any] | None = None
if model_available:
    stream_result = stream_generate(
        "Explain in two sentences why API timeouts matter.",
        temperature=0.2,
    )
    print(stream_result["text"])
    print(
        {
            "first_content_seconds": stream_result["first_content_seconds"],
            "total_wall_seconds": stream_result["total_wall_seconds"],
            "done_reason": stream_result["done_reason"],
        }
    )
else:
    print("Live streaming skipped because the configured model is unavailable.")

### Interpret Streaming Measurements

Repeat the request before drawing a conclusion. Model loading can dominate the first run, and prompt length affects processing time. Record the model tag, hardware, prompt, settings, cold or warm state, and repetitions alongside the result.

A lower time to first content can improve perceived responsiveness even when the complete response takes the same total time.

## 4. Define a JSON Schema

Structured output is useful when another component needs predictable fields. This example extracts a category, urgency, and short summary from synthetic support text.

The schema controls structure and allowed values. It does not verify whether the chosen category or urgency is correct.

In [ ]:
SUPPORT_SCHEMA = {
    "type": "object",
    "properties": {
        "category": {
            "type": "string",
            "enum": ["billing", "technical", "account"],
        },
        "urgency": {
            "type": "string",
            "enum": ["low", "medium", "high"],
        },
        "summary": {"type": "string", "minLength": 1},
    },
    "required": ["category", "urgency", "summary"],
    "additionalProperties": False,
}

fixed_structured_output = {
    "category": "account",
    "urgency": "medium",
    "summary": "The user cannot reset their password.",
}
validate(instance=fixed_structured_output, schema=SUPPORT_SCHEMA)
print("Fixed example matches the schema.")

## 5. Request and Validate Structured Output

Ollama's generate endpoint accepts a JSON Schema in the `format` field. The prompt should still describe the task clearly. The client then performs three separate checks:

1. **HTTP success:** the service returned a successful response;
2. **JSON parsing:** the response text is syntactically valid JSON; and
3. **schema validation:** required fields, types, and allowed values match.

A fourth check, semantic evaluation, remains necessary for content correctness.

In [ ]:
def generate_structured(
    source_text: str,
    *,
    model: str = MODEL_NAME,
) -> dict[str, Any]:
    """Generate, parse, and schema-validate a support classification."""
    prompt = (
        "Classify this synthetic support request. Return only JSON matching "
        f"the supplied schema. Request: {source_text}"
    )
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "think": False,
        "format": SUPPORT_SCHEMA,
        "options": {
            "temperature": 0,
            "num_predict": MAX_OUTPUT_TOKENS,
        },
    }

    response = session.post(
        GENERATE_URL,
        json=payload,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    raw_text = str(response.json().get("response", ""))
    parsed = json.loads(raw_text)
    validate(instance=parsed, schema=SUPPORT_SCHEMA)
    return parsed

In [ ]:
if model_available:
    try:
        structured_result = generate_structured(
            "I am locked out after changing my password twice."
        )
        print(json.dumps(structured_result, indent=2))
    except (requests.RequestException, json.JSONDecodeError, ValidationError) as exc:
        print(f"Structured generation failed validation: {exc}")
else:
    print("Live structured generation skipped because the model is unavailable.")

## 6. Test Failure Handling

Validation errors are expected application states, not exceptional surprises to hide. The fixed example below demonstrates a response that parses as JSON but fails the schema because the category is unsupported and a required field is missing.

In [ ]:
invalid_output = {
    "category": "sales",
    "summary": "The user asks about an upgrade.",
}

try:
    validate(instance=invalid_output, schema=SUPPORT_SCHEMA)
except ValidationError as exc:
    print(f"Validation correctly rejected the output: {exc.message}")

## 7. Practice

Adapt the code without introducing sensitive data:

1. Change the schema for a different technical task, such as extracting a bug title, severity, and affected component.
2. Add one valid and two invalid fixed examples before calling the model.
3. Run the model several times and record parse failures, schema failures, and semantic errors separately.
4. Compare streaming and non-streaming client timing for the same prompt.
5. Decide which failures should trigger retry, correction, user review, or rejection.

Keep the task narrow and change one condition at a time.

## Summary

You implemented two production-facing API patterns:

- streaming exposes partial output and allows client-observed first-content timing;
- JSON Schema constrains output structure and enables deterministic validation.

Neither pattern proves semantic correctness. Reliable integration still needs representative test cases, explicit failure handling, and human review where consequences are significant.

## References

- [Ollama API: Streaming](https://docs.ollama.com/api/streaming)
- [Ollama API: Generate a Response](https://docs.ollama.com/api/generate)
- [Ollama Thinking](https://docs.ollama.com/capabilities/thinking)
- [Ollama API: Structured Outputs](https://docs.ollama.com/capabilities/structured-outputs)
- [JSON Schema Documentation](https://json-schema.org/learn/getting-started-step-by-step)